In [ ]:
import sys
print(sys.path)

In [ ]:
import os
print(os.getcwd())

In [1]:
!python --version

Python 3.13.9


In [2]:
!pip install mcp

In [5]:
!mcp --help

                                                                               
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                        
                                                                               
 MCP development tools                                                         
                                                                               
+- Options -------------------------------------------------------------------+
| --help          Show this message and exit.                                 |
+-----------------------------------------------------------------------------+
+- Commands ------------------------------------------------------------------+
| version   Show the MCP version.                                             |
| dev       Run an MCP server with the MCP Inspector.                         |
| run       Run an MCP server.                                                |
| install   Install an MCP server in the

In [6]:
!mcp version

MCP version 1.27.0


In [7]:
#Actual project starts from here
from typing import Any

import httpx
from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("weather")

# Constants
NWS_API_BASE = "https://api.weather.gov"
USER_AGENT = "weather-app/1.0"

In [8]:
#Adding helper function for querying and formatting the data from the national weather service
async def make_nws_request(url: str) -> dict[str, Any] | None:
    """Make a request to the NWS API with proper error handling."""
    headers = {"User-Agent": USER_AGENT, "Accept": "application/geo+json"}
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(url, headers=headers, timeout=30.0)
            response.raise_for_status()
            return response.json()
        except Exception:
            return None


def format_alert(feature: dict) -> str:
    """Format an alert feature into a readable string."""
    props = feature["properties"]
    return f"""
Event: {props.get("event", "Unknown")}
Area: {props.get("areaDesc", "Unknown")}
Severity: {props.get("severity", "Unknown")}
Description: {props.get("description", "No description available")}
Instructions: {props.get("instruction", "No specific instructions provided")}
"""

In [9]:
#The tool execution handler 
@mcp.tool()
async def get_alerts(state: str) -> str:
    """Get weather alerts for a US state.

    Args:
        state: Two-letter US state code (e.g. CA, NY)
    """
    url = f"{NWS_API_BASE}/alerts/active/area/{state}"
    data = await make_nws_request(url)

    if not data or "features" not in data:
        return "Unable to fetch alerts or no alerts found."

    if not data["features"]:
        return "No active alerts for this state."

    alerts = [format_alert(feature) for feature in data["features"]]
    return "\n---\n".join(alerts)


@mcp.tool()
async def get_forecast(latitude: float, longitude: float) -> str:
    """Get weather forecast for a location.

    Args:
        latitude: Latitude of the location
        longitude: Longitude of the location
    """
    # First get the forecast grid endpoint
    points_url = f"{NWS_API_BASE}/points/{latitude},{longitude}"
    points_data = await make_nws_request(points_url)

    if not points_data:
        return "Unable to fetch forecast data for this location."

    # Get the forecast URL from the points response
    forecast_url = points_data["properties"]["forecast"]
    forecast_data = await make_nws_request(forecast_url)

    if not forecast_data:
        return "Unable to fetch detailed forecast."

    # Format the periods into a readable forecast
    periods = forecast_data["properties"]["periods"]
    forecasts = []
    for period in periods[:5]:  # Only show next 5 periods
        forecast = f"""
{period["name"]}:
Temperature: {period["temperature"]}°{period["temperatureUnit"]}
Wind: {period["windSpeed"]} {period["windDirection"]}
Forecast: {period["detailedForecast"]}
"""
        forecasts.append(forecast)

    return "\n---\n".join(forecasts)

In [16]:
# Execute the demo
result = await get_alerts('MI')
print(f"\n{result}")

[05/08/26 12:40:23] INFO     HTTP Request: GET https://api.weather.gov/alerts/active/area/MI        ]8;id=983954;file://C:\Users\shawo\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=40101;file://C:\Users\shawo\anaconda3\Lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     



Event: Special Weather Statement
Area: Emmet; Cheboygan; Presque Isle; Leelanau; Antrim; Otsego; Montmorency; Alpena; Benzie; Grand Traverse; Kalkaska; Crawford; Oscoda; Alcona; Manistee; Wexford; Missaukee; Roscommon; Ogemaw; Iosco; Gladwin; Arenac; Beaver Island and surrounding islands; Charlevoix
Severity: Moderate
Description: A combination of drying surface conditions, very low afternoon
relative humidity values, and at times gusty west to southwest
winds will result in elevated fire danger conditions across
northern lower Michigan today. Sustained winds of 10 to 15 mph
with occasional gusts of 20 to 30 mph will be seen over northern
Michigan late this morning through this afternoon. Minimum
relative humidity values of 20 to 30 percent are expected over
northern lower Michigan. These conditions could lead to rapid
spread of fires. Check burning restrictions and fire danger before
burning.

For more information on burning restrictions for Michigan, see
www.dnr.state.mi.us//.
Inst